# PSDL: Try It in 2 Minutes

**PSDL (Patient Scenario Definition Language)** is an open, vendor-neutral standard for expressing clinical detection logic — portable, auditable, and deterministic.

In this notebook you will:
1. Load the bundled **KDIGO AKI staging** scenario (the flagship example)
2. Generate three synthetic patients and evaluate them
3. Look under the hood: compilation hashes and the auto-generated **SQL** for cohort-scale execution

> Same YAML, three execution modes: single-patient (Python), cohort (SQL), streaming (Flink).


## Step 1: Install


In [ ]:
%pip install -q psdl-lang==0.5.1 matplotlib requests


In [ ]:
from datetime import datetime, timedelta

import psdl
from psdl import compile_scenario, examples, parse_scenario
from psdl.runtimes.single import InMemoryBackend, SinglePatientEvaluator

print(f"psdl-lang {psdl.__version__} ready")


## Step 2: The scenario

A PSDL scenario has four required parts: **audit** (why), **signals** (what data), **trends** (temporal math), **logic** (boolean decisions).

`aki_detection.yaml` implements the creatinine arm of the KDIGO criteria. Note that the baseline is computed *in-language* — `min(Cr, 7d)` — and the stages are mutually exclusive via `NOT` (highest stage wins).


In [ ]:
scenario = parse_scenario(examples.get_scenario_path("aki_detection"))
print(f"{scenario.name} v{scenario.version}\n")

print("Signals:")
for name, sig in scenario.signals.items():
    print(f"  {name}: {sig.ref} ({sig.unit})")

print("\nTrends (numeric):")
for name, trend in scenario.trends.items():
    print(f"  {name} = {trend.raw_expr}")

print("\nLogic (boolean):")
for name, logic in scenario.logic.items():
    sev = f" [{logic.severity.value}]" if logic.severity else ""
    print(f"  {name}{sev}: {logic.expr}")


## Step 3: Three synthetic patients


In [ ]:
now = datetime(2026, 8, 1, 12, 0)

patients = {
    "stable":     [(-5, 0.90), (-3, 1.00), (-1, 0.95), (0, 0.92)],
    "aki_stage2": [(-6, 1.00), (-4, 1.10), (0, 2.50)],   # 2.5x baseline
    "aki_stage3": [(-6, 1.00), (-2, 2.20), (0, 4.50)],   # >=3x baseline AND Cr >= 4.0
}
bun = {"stable": 15.0, "aki_stage2": 32.0, "aki_stage3": 40.0}

backend = InMemoryBackend()
for pid, series in patients.items():
    for days, value in series:
        backend.add_observation(pid, "Cr", value, now + timedelta(days=days))
    backend.add_observation(pid, "BUN", bun[pid], now)

print(f"Loaded {backend.observation_count()} observations for {len(patients)} patients")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for pid, series in patients.items():
    xs = [days for days, _ in series]
    ys = [v for _, v in series]
    ax.plot(xs, ys, marker="o", label=pid)
ax.axhline(4.0, color="red", ls="--", lw=1, label="KDIGO Stage 3 absolute (4.0)")
ax.set_xlabel("days from evaluation time")
ax.set_ylabel("creatinine (mg/dL)")
ax.legend()
ax.set_title("Synthetic creatinine trajectories")
plt.show()


## Step 4: Evaluate

The evaluator computes every trend (including arithmetic expressions like `last(Cr) / min(Cr, 7d)`) and resolves the logic DAG. Missing data propagates as `None -> False`, never as a silent guess.


In [ ]:
stage_cols = ["aki_stage1", "aki_stage2", "aki_stage3", "aki_present"]

print(f"{'patient':<12} {'cr_ratio':>8} {'rise48h':>8}  stages")
for pid in patients:
    r = SinglePatientEvaluator(scenario, backend).evaluate(pid, reference_time=now)
    ratio = r.trend_values.get("cr_ratio")
    rise = r.trend_values.get("cr_rise_48h")
    stages = [c for c in stage_cols if r.logic_results.get(c)]
    ratio_s = f"{ratio:.2f}" if ratio is not None else "-"
    rise_s = f"{rise:.2f}" if rise is not None else "-"
    print(f"{pid:<12} {ratio_s:>8} {rise_s:>8}  {stages or 'none'}")


## Step 5: Under the hood — compilation and hashes

`compile_scenario()` produces a **ScenarioIR**: dependency-ordered evaluation graph plus canonical hashes (RFC-0006). Two institutions running the same scenario can prove they ran *exactly* the same logic.


In [ ]:
ir = compile_scenario(examples.get_scenario_path("aki_detection"))
print(f"ir_hash:   {ir.ir_hash}")
print(f"spec_hash: {ir.spec_hash}")
print(f"compiled:  {ir.compiled_at}")
print(f"warnings:  {ir.compilation.warnings or 'none'}")

print("\nSignal groups (v0.5, RFC-0009):")
for name, grp in scenario.signal_groups.items():
    print(f"  {name}: members={grp.members} — {grp.description}")


## Step 6: The same scenario as SQL

For cohort-scale retrospective runs, the **CohortCompiler** turns the scenario into a single SQL query. Physical bindings (tables, concept IDs) come from a separate **Dataset Spec** — the scenario itself stays vendor-neutral.

This exact pipeline was validated on a 288M-row MIMIC-IV/OMOP database (294,788 patients), with a 200-patient consistency check against the Python evaluator: **zero mismatches**.


In [ ]:
import pathlib
import requests

spec_url = ("https://raw.githubusercontent.com/Chesterguan/PSDL/main/"
            "dataset_specs/omop_cdm_v54.yaml")
pathlib.Path("omop_cdm_v54.yaml").write_text(requests.get(spec_url).text)

from psdl import load_dataset_spec
from psdl.runtimes.cohort import CohortCompiler

ds = load_dataset_spec("omop_cdm_v54.yaml", schema="omop_cdm")
compiled = CohortCompiler(schema="omop_cdm").compile(scenario, dataset_spec=ds)

print("\n".join(compiled.sql.splitlines()[:30]))
print(f"\n... ({len(compiled.sql.splitlines())} lines total)")


## Where to go next

- **Repository & spec**: https://github.com/Chesterguan/PSDL
- **MIMIC-IV demo notebook**: real ICU patients with ICD ground truth (`PSDL_MIMIC_Demo.ipynb`)
- **Streaming demo**: event-by-event evaluation (`PSDL_Streaming_Demo.ipynb`)
- **PSDL Inspector**: describe a scenario in one sentence, get a validated, audit-ready algorithm — https://github.com/Chesterguan/psdl-inspector
